# M5 Forecasting Competition EDA

This notebook explores key ideas for understanding the M5 Forecasting Competition data, which will help in crafting a Chronos-2 zero-shot baseline.

# Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

FileNotFoundError: [Errno 2] No usable temporary directory found in ['/tmp', '/var/tmp', '/usr/tmp', '/home/nmwamsojo/tsfm-explo/notebooks']

# Config

# Load and format the sales data

# Utility functions

In [ ]:


# Tags to isolate experiments
dataprep_tags = "baseline" # data tags for data preparation choices
modeling_tags = "baseline" # modeling tags for modeling
postproc_tags = "baseline" # postprocessing tags for postprocessing

cutoff_day = "2016-05-22" # "2016-03-27"

new_cutoff = pd.to_datetime(cutoff_day) - pd.Timedelta(days=0)
#cutoff_day = new_cutoff.strftime('%Y-%m-%d')
print(f"Cutoff day: {cutoff_day}")

dataprep_config = {
    "tag": "default",
    "path": "",
    "hist_cols": [],
    "fut_cols": [],  # for synamic covariates among ["wm_yr_wk", "wday", "month", "year", "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI", "sell_price"]
    "stat_cols": ["item_id", "dept_id", "cat_id", "store_id", "state_id"], # for static covariates among ["item_id", "dept_id", "cat_id", "store_id", "state_id"]
}

Cutoff day: 2016-05-22


In [ ]:
import os
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

class M5DataPipeline:
    def __init__(self, config, target_col="sales_quantity"):
        self.target = target_col
        self.date = "date"
        self.id = "id"
        self.base_dir = "/mnt/lab/nmwamsojo/prepared_data"
        self.tag = config.get("tag", "default")
        
        # New: Track the model tag for the "skip" logic on forecasts
        self.model_tag = config.get("model_tag", "default_model")

        self.level_map = {
            1: [], 2: ["state_id"], 3: ["store_id"], 4: ["cat_id"], 5: ["dept_id"],
            6: ["state_id", "cat_id"], 7: ["state_id", "dept_id"], 8: ["store_id", "cat_id"],
            9: ["store_id", "dept_id"], 10: ["item_id"], 11: ["state_id", "item_id"], 12: ["id"]
        }
        self.covariates = ["wm_yr_wk", "wday", "month", "year", "event_name_1", "event_type_1", "snap_CA", "snap_TX", "snap_WI", "sell_price"]
        self.static_cols = ["item_id", "dept_id", "cat_id", "store_id", "state_id"]

    def _get_base_folder(self, cutoff_day, level):
        """Returns the specific experiment leaf folder."""
        return os.path.join(
            self.base_dir, 
            self.tag, 
            f"level_{level}", 
            cutoff_day.replace("-", "")
        )

    def _get_cache_paths(self, cutoff_day, level):
        """Paths for prepared data."""
        folder = self._get_base_folder(cutoff_day, level)
        return {
            "folder": folder,
            "hist": os.path.join(folder, "hist.parquet"),
            "future": os.path.join(folder, "future.parquet"),
            "static": os.path.join(folder, "static.parquet")
        }

    def get_forecast_paths(self, cutoff_day, level, model_tag=None):
        """Paths for model-specific outputs within the same experiment folder."""
        m_tag = model_tag or self.model_tag
        folder = os.path.join(self._get_base_folder(cutoff_day, level), "models", m_tag)
        return {
            "folder": folder,
            "forecast": os.path.join(folder, "forecasts.parquet"),
            "metrics": os.path.join(folder, "metrics.json")
        }


    
    def prepare(self, path: str, cutoff_day: str, level: int = 12):
        print(f"--- Loading: {path} ---")
        aggr_cols = self.level_map[level]

        # 1. SMART LOADING: Check schema to only load what exists
        schema_names = set(pq.read_schema(path).names) # TODO: remove
        target_on_disk = "sold" if "sold" in schema_names else self.target
        
        # Build the 'needed' set dynamically based on what's actually in the file
        needed = {self.id, self.date, target_on_disk} | \
                 (set(aggr_cols) & schema_names) | \
                 (set(self.covariates) & schema_names) | \
                 (set(self.static_cols) & schema_names)
        
        df = pd.read_parquet(path, columns=list(needed))

        if target_on_disk != self.target:
            df.rename(columns={target_on_disk: self.target}, inplace=True)

        df[self.date] = pd.to_datetime(df[self.date], utc=False, cache=True)

        # 2. OPTIMIZED DTYPES & CLEANING
        # Event columns: Handle Categorical constraints without full re-casting
        for col in [c for c in df.columns if "event" in c]:
            if hasattr(df[col], "cat"):
                if "none" not in df[col].cat.categories:
                    df[col] = df[col].cat.add_categories("none")
                df[col] = df[col].fillna("none")
            else:
                df[col] = df[col].fillna("none").astype("category")

        # Downcast with copy=False to save RAM if dtypes already match
        # Logic: int8 for flags, int16 for calendar parts, float32 for values
        type_map = {
            "snap": np.int8, "wday": np.int8, "month": np.int8,
            "year": np.int16, "wm_yr_wk": np.int16, 
            "sell_price": np.float32, self.target: np.float32
        }
        
        for col, dtype in type_map.items():
            # Check for partial matches like 'snap_'
            match_cols = [c for c in df.columns if col in c]
            for c in match_cols:
                df[c] = df[c].astype(dtype)

        # Ensure grouping/static columns are Categorical for speed
        for col in aggr_cols + self.static_cols:
            if col in df.columns and df[col].dtype == object:
                df[col] = df[col].astype("category")

        # 3. AGGREGATION
        print(f"--- Aggregating to Level {level} ---")
        agg_map = {self.target: "sum"}
        
        for col in self.covariates:
            if col in df.columns:
                if "snap" in col: agg_map[col] = "max"
                elif "event" in col: agg_map[col] = "first"
                else: agg_map[col] = "mean"

        for col in self.static_cols:
            if col in df.columns and col not in aggr_cols:
                agg_map[col] = "first" 

        df_aggr = (
            df.groupby(aggr_cols + [self.date], observed=True, sort=False)
            .agg(agg_map)
            .reset_index()
        )
        del df # memory release

        # 4. FAST ID GENERATION
        if len(aggr_cols) == 0:
            df_aggr[self.id] = "All"
        elif len(aggr_cols) == 1:
            df_aggr[self.id] = df_aggr[aggr_cols[0]].astype(str)
        else:
            # High-speed categorical-to-string decoding
            def _get_str_vals(s):
                if hasattr(s, "cat"):
                    return s.cat.categories.astype(str).values[s.cat.codes.values]
                return s.astype(str).values

            parts = [_get_str_vals(df_aggr[c]) for c in aggr_cols]
            ids = parts[0]
            for part in parts[1:]:
                ids = np.char.add(np.char.add(ids, "_"), part)
            df_aggr[self.id] = ids

        df_aggr[self.id] = df_aggr[self.id].str.replace("_evaluation", "", regex=False).astype("category")

        # 5. SPLITTING
        cutoff = pd.Timestamp(cutoff_day)
        df_aggr = df_aggr.sort_values([self.id, self.date]).reset_index(drop=True)

        mask = df_aggr[self.date] <= cutoff

        hist_df = df_aggr[mask].copy()
        future_df = df_aggr[~mask].copy()
        if self.target in future_df.columns:
            future_df.drop(columns=[self.target], inplace=True)
        
        # 6. STATIC METADATA
        static_present = [c for c in self.static_cols if c in df_aggr.columns]
        static_df = df_aggr[[self.id] + static_present].drop_duplicates().reset_index(drop=True)

        print(f"Done. {df_aggr[self.id].nunique():,} series | "
              f"Hist: {len(hist_df):,} rows | Future: {len(future_df):,} rows")
        del df_aggr # memory release
        return hist_df, future_df, static_df
    def get_prepared_data(self, path, cutoff_day, level=12, force_reprepare=False):

        paths = self._get_cache_paths(cutoff_day, level)

        # determine if data exists and if we should use it given paths above and force_reprepare flag
        use_cache = os.path.exists(paths["hist"]) and not force_reprepare

        if use_cache:
            print(f"--- Cache Hit: Data found in {paths['folder']} ---")
            return (pd.read_parquet(paths["hist"]), 
                    pd.read_parquet(paths["future"]), 
                    pd.read_parquet(paths["static"]))
        
        # for tracking purposes
        reason = "Force Reprepare" if force_reprepare else "Cache Miss"
        print(f"--- {reason}: Preparing Level {level} for {cutoff_day} ---")

        hist_df, future_df, static_df = self.prepare(path, cutoff_day, level)
        os.makedirs(paths["folder"], exist_ok=True)
        hist_df.to_parquet(paths["hist"], index=False)
        future_df.to_parquet(paths["future"], index=False)
        static_df.to_parquet(paths["static"], index=False)
        return hist_df, future_df, static_df

    def get_forecasts(self, model_func, path, cutoff_day, level=12, force_train=False):
        """
        Implements skip logic for the modeling phase.
        model_func: a function that takes (hist, future, static) and returns forecasts_df
        """
        f_paths = self.get_forecast_paths(cutoff_day, level)
        
        if os.path.exists(f_paths["forecast"]) and not force_train:
            print(f"--- Model Cache Hit: Loading forecasts from {f_paths['folder']} ---")
            return pd.read_parquet(f_paths["forecast"])

        # Otherwise, we need data
        hist_df, future_df, static_df = self.get_prepared_data(path, cutoff_day, level)
        
        print(f"--- Model Cache Miss: Training/Inference for {self.model_tag} ---")
        forecasts_df = model_func(hist_df, future_df, static_df)
        
        # Save results
        os.makedirs(f_paths["folder"], exist_ok=True)
        forecasts_df.to_parquet(f_paths["forecast"], index=False)
        print(f"--- Forecasts saved to {f_paths['forecast']} ---")
        
        return forecasts_df

# Initialize
pipeline = M5DataPipeline(config=dataprep_config)
path = "/mnt/lab/datasets/M5/jointed_M5.parquet"

# CALL THE CACHE WRAPPER, NOT PREPARE DIRECTLY
hist_df, future_df, static_df = pipeline.get_prepared_data(path, cutoff_day, level=12) #, force_reprepare=True)


--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/default/level_12/20160522 ---


## Actual data for evaluations

In [ ]:
import pandas as pd

def transform_m5_ground_truth(gt_wide_df, calendar_path="/mnt/lab/nmwamsojo/m5_data/calendar.csv"):
    """
    Converts wide M5 ground truth (d_1942...) to long format (id, date, sales_quantity)
    """
    # 1. Create the 'id' column if it doesn't exist (joining item and store)
    if 'id' not in gt_wide_df.columns:
        gt_wide_df['id'] = gt_wide_df['item_id'] + '_' + gt_wide_df['store_id'] + '_evaluation'
    
    # 2. Melt the dataframe from wide to long
    day_cols = [c for c in gt_wide_df.columns if c.startswith('d_')]
    df_long = gt_wide_df.melt(
        id_vars=['id'],
        value_vars=day_cols,
        var_name='d',
        value_name='sales_quantity'
    )
    
    # 3. Map 'd_xxxx' to actual datetime objects using calendar
    calendar = pd.read_csv(calendar_path)
    calendar['date'] = pd.to_datetime(calendar['date'])
    
    # Create a mapping dictionary for speed
    d_to_date = dict(zip(calendar['d'], calendar['date']))
    df_long['date'] = df_long['d'].map(d_to_date)
    
    # 4. Final cleanup
    # Ensure ID matches your forecast format (usually removing the suffix)
    df_long['id'] = df_long['id'].str.replace('_evaluation', '', regex=False)
    
    return df_long[['id', 'date', 'sales_quantity']]

# --- Usage ---
# Assuming you find a file with the actuals for d_1942-d_1969
actuals_eval = pd.read_csv("/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv") 

# actual eval for final metrics
df_actual_eval = transform_m5_ground_truth(actuals_eval)

# read true sales for evaluation (can be used later in the notebook)
df_actual = pd.read_parquet(path, columns=['id', 'date', 'sold']).rename(columns={'sold': 'sales_quantity'})
df_actual["id"] = (
    df_actual["id"]
    .astype(str)
    .str.replace("_evaluation", "", regex=False)
    .str.replace("_validation", "", regex=False)
)



# Modeling

In [ ]:
import pandas as pd
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

class AutoGluonESForecaster:
    def __init__(self, horizon=28, model_path="/mnt/lab/nmwamsojo/autogluon_models/m5_es_bu"):
        self.horizon = horizon
        self.target = "sales_quantity"
        self.model_save_path = model_path

    def fit_predict(self, hist_df, static_df=None):
        print(f"--- Converting to AutoGluon format ---")
        
        # Ensure date is datetime and sort for AutoGluon
        hist_df["date"] = pd.to_datetime(hist_df["date"])
        hist_df = hist_df.sort_values(["id", "date"]).reset_index(drop=True)
        
        # 1. Create the TimeSeriesDataFrame
        ag_df = TimeSeriesDataFrame.from_data_frame(
            hist_df,
            id_column="id",
            timestamp_column="date"
        )
        
        # 2. Add static features correctly
        if static_df is not None:
            # Drop 'id' or 'item_id' from columns to prevent duplicate column errors 
            # when AutoGluon resets the index internally.
            cols_to_drop = [c for c in ["id", "item_id"] if c in static_df.columns]
            temp_static = static_df.set_index("id").drop(columns=cols_to_drop, errors='ignore')
            
            # Match the dtype of the TimeSeriesDataFrame index (usually object or category)
            temp_static.index = temp_static.index.astype(ag_df.item_ids.dtype)
            
            # The name of the index must NOT exist in the columns. 
            # AutoGluon uses 'item_id' by default internally.
            temp_static.index.name = "item_id" 
            
            ag_df.static_features = temp_static.loc[ag_df.item_ids]

        print(f"--- Fitting ETS (ANN) ---")
        # ES_bu reproduction: 'ANN' = Simple Exponential Smoothing
        custom_hyperparameters = {
            "ETS": {
                "model": "ANN",
                "seasonal_period": None 
            }
        }

        predictor = TimeSeriesPredictor(
            prediction_length=self.horizon,
            target=self.target,
            eval_metric="RMSE",
            path=self.model_save_path
        ).fit(
            ag_df,
            hyperparameters=custom_hyperparameters,
            enable_ensemble=False,
            skip_model_selection=True,
            verbosity=1
        )

        print(f"--- Generating Forecast ---")
        # Use the same ag_df to ensure index alignment
        predictions = predictor.predict(ag_df)
        
        # 3. Convert MultiIndex back to Long Format
        f_df = predictions['mean'].reset_index()
        # AutoGluon output index name is 'item_id' by default
        f_df.columns = ["id", "date", self.target]
        
        return f_df

    def aggregate_bu(self, f_df, static_df, level_map):
        """
        Aggregates Level 12 forecasts up to all M5 levels.
        """
        print("--- Aggregating Bottom-Up ---")
        # Ensure dtypes match for the merge
        f_df["id"] = f_df["id"].astype(str)
        static_df_copy = static_df.copy()
        static_df_copy["id"] = static_df_copy["id"].astype(str)
        
        merged_f = f_df.merge(static_df_copy, on="id", how='left')
        
        all_levels = []
        for lvl, group_cols in level_map.items():
            if lvl == 12:
                all_levels.append(f_df[["id", "date", self.target]])
                continue
                
            if not group_cols: # Level 1 (Total)
                agg = merged_f.groupby("date")[self.target].sum().reset_index()
                agg["id"] = "Total"
            else:
                agg = merged_f.groupby(group_cols + ["date"], observed=True)[self.target].sum().reset_index()
                # Create ID by joining categorical values (e.g., 'CA_FOODS')
                agg["id"] = agg[group_cols].astype(str).agg('_'.join, axis=1)
                
            all_levels.append(agg[["id", "date", self.target]])
            
        return pd.concat(all_levels, axis=0).reset_index(drop=True)

FileNotFoundError: [Errno 2] No usable temporary directory found in ['/tmp', '/var/tmp', '/usr/tmp', '/home/nmwamsojo/tsfm-explo/notebooks']

In [ ]:
predictor = AutoGluonESForecaster()
#level_forecast_df_ag = predictor.fit_predict(hist_df, static_df)

--- Converting to AutoGluon format ---


Renaming existing column 'item_id' -> '__item_id' to avoid name collisions.


--- Fitting ETS (Exponential Smoothing) & Saving to: /mnt/lab/nmwamsojo/autogluon_models/m5_benchmark ---
--- Generating Forecast ---


# Post-processing

# Evaluation

In [ ]:
import numpy as np
import pandas as pd


class M5Evaluator:
    def __init__(self, train_df: pd.DataFrame, static_df: pd.DataFrame, target_col: str = "sales_quantity",
                 price_col: str = "sell_price"):
        self.target = target_col

        # .copy() — avoids mutating the caller's frame when we set_index
        self._static = static_df.set_index("id").copy()
        self._static_cols = list(self._static.columns)

        self.levels = {
            1: [], 2: ["state_id"], 3: ["store_id"], 4: ["cat_id"], 5: ["dept_id"],
            6: ["state_id", "cat_id"], 7: ["state_id", "dept_id"], 8: ["store_id", "cat_id"],
            9: ["store_id", "dept_id"], 10: ["item_id"], 11: ["state_id", "item_id"], 12: ["id"]
        }

        self._price_col = price_col

        print("--- Pre-calculating Hierarchy Scales and Weights ---")

        # Pivot training data; sort columns to guarantee date order (required for scale calc)
        train_wide = (
            train_df.pivot(index="id", columns="date", values=target_col)
                    .fillna(0)
                    .sort_index(axis=1)
        )

        self.date_cols = train_wide.columns

        self.ids = train_wide.index.astype(str)
        self._static.index = self._static.index.astype(str)

        # Revenue wide: quantity x price, same shape as train_wide.
        # sell_price is sparse per item — forward/back-fill before multiply.
        # Falls back to unit volume if price_col absent.
        if price_col in train_df.columns:
            price_wide = (
                train_df.pivot(index="id", columns="date", values=price_col)
                        .reindex(index=train_wide.index, columns=train_wide.columns)
            )
            # ffill then bfill handles sparse price observations; 1.0 covers any remaining NaN
            price_wide = price_wide.ffill(axis=1).bfill(axis=1).fillna(1.0)
            revenue_wide = train_wide * price_wide   # elementwise: units * $/unit = $
            del price_wide
        else:
            print(f"[M5Evaluator] WARNING: '{price_col}' not in train_df — falling back to volume weights.")
            revenue_wide = train_wide.copy()

        self.level_info = self._prepare_hierarchy(train_wide, revenue_wide)

        # Free both matrices — ~460 MB total — only level_info is needed from here
        del train_wide, revenue_wide

    # ------------------------------------------------------------------ #
    #  Hierarchy pre-computation                                           #
    # ------------------------------------------------------------------ #

    def _prepare_hierarchy(self, train_wide: pd.DataFrame, revenue_wide: pd.DataFrame) -> dict:
        df          = train_wide.join(self._static)
        rev_df      = revenue_wide.join(self._static)
        date_cols   = list(train_wide.columns)

        level_info = {}
        for lvl, cols in self.levels.items():
            aggr     = self._aggregate(df,     date_cols, cols, lvl)
            rev_aggr = self._aggregate(rev_df, date_cols, cols, lvl)

            scales = self._calculate_scales(aggr.values)

            # Weights by revenue ($ volume) over last 28 training days — matches
            # the official M5 competition weighting which uses dollar sales, not units
            w_vals  = rev_aggr.values[:, -28:].sum(axis=1)
            weights = w_vals / (w_vals.sum() + 1e-8)

            level_info[lvl] = {
                "scales":  scales,
                "weights": weights,
                "cols":    cols,
                "index":   aggr.index,
                "date_cols": date_cols,   # stored so evaluate_all can align safely
            }
        return level_info

    @staticmethod
    def _aggregate(df: pd.DataFrame, date_cols: list, cols: list, lvl: int) -> pd.DataFrame:
        """Aggregate a wide+static frame to a hierarchy level, returning only date columns."""
        if lvl == 1:
            # Total: single row
            return df[date_cols].sum().to_frame().T
        return df.groupby(cols, observed=True)[date_cols].sum()

    # ------------------------------------------------------------------ #
    #  Scale calculation — fully vectorised, no Python row loop           #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _calculate_scales(array: np.ndarray) -> np.ndarray:
        """
        Vectorised RMSSE denominator (mean squared first-difference of non-zero tail).

        Strategy: compute diff² for every row simultaneously, then mask out the
        leading-zero prefix per row using a cumulative-max trick so we never loop.
        """
        n_rows, n_cols = array.shape

        # --- mask leading zeros vectorised ---
        # nonzero[r, t] = True once the first nonzero value has appeared in row r
        nonzero_mask = np.maximum.accumulate(array > 0, axis=1)   # (n_rows, n_cols)

        # first-differences; pad one False column so diff has same width as mask
        diffs = np.diff(array, axis=1)                             # (n_rows, n_cols-1)
        diff_mask = nonzero_mask[:, 1:]                            # align to diffs

        # squared diffs, zeroed where still in the leading-zero prefix
        sq_diffs = (diffs ** 2) * diff_mask                        # (n_rows, n_cols-1)

        # mean over valid (non-prefix) positions per row
        valid_counts = diff_mask.sum(axis=1).clip(min=1)           # avoid /0
        scales = sq_diffs.sum(axis=1) / valid_counts

        return np.maximum(scales, 1e-8)                            # floor to avoid /0 in RMSSE

    # ------------------------------------------------------------------ #
    #  Public evaluation                                                   #
    # ------------------------------------------------------------------ #
    def get_detailed_report(self, forecast_df, actual_df):
        """Returns a DataFrame showing WRMSSE, Weight, and Scale-mean for each level."""
        # Run the internal logic of evaluate_all


        # filter 
        actual_df = actual_df[(actual_df["date"]>=forecast_df["date"].min()) & (actual_df["date"]<=forecast_df["date"].max())].copy()
        #actual_df["id"] = actual_df["id"].astype(self.ids.dtype)
        #forecast_df = forecast_df.copy()
        forecast_df["id"] = forecast_df["id"].astype(self.ids.dtype)

        # ── Pivot ────────────────────────────────────────────────────────
        # Use self.target explicitly — never rely on column position
        f_wide = forecast_df.pivot(index="id", columns="date", values=self.target)
        a_wide = actual_df.pivot(  index="id", columns="date", values=self.target)

        common_dates = a_wide.columns.intersection(f_wide.columns)

        # Warn loudly on partial overlap — silent data loss is worse than a warning
        if len(common_dates) < len(a_wide.columns):
            missing = len(a_wide.columns) - len(common_dates)
            print(f"[M5Evaluator] WARNING: {missing} actual date(s) have no matching forecast — treated as 0.")
        if len(common_dates) == 0:
            raise ValueError("forecast_df and actual_df share no common dates.")

        # Validate series alignment — reindex fills with 0 silently, so check first
        missing_ids = set(self.ids) - set(f_wide.index)
        if missing_ids:
            print(f"[M5Evaluator] WARNING: {len(missing_ids)} series missing from forecast — filled with 0.")

        # Align to training IDs and common dates; fillna(0) only after explicit warning above
        f12 = f_wide.reindex(index=self.ids, columns=common_dates).fillna(0)
        a12 = a_wide.reindex(index=self.ids, columns=common_dates).fillna(0)

        # Join static once; use explicit column names from here on — no positional slicing
        f12 = f12.join(self._static)
        a12 = a12.join(self._static)
        assert a12[self._static_cols].notna().any().any(), "Static join produced all NaN — index dtype mismatch between a_wide and self._static"
        date_cols = list(common_dates)

        # ── WRMSSE across 12 levels ──────────────────────────────────────
        level_scores = []
        for lvl in range(1, 13):
            info = self.level_info[lvl]

            # Aggregate using explicit date_cols — immune to column order changes
            f_lvl = self._aggregate(f12, date_cols, info["cols"], lvl).values
            a_lvl = self._aggregate(a12, date_cols, info["cols"], lvl).values

            mse        = np.mean((a_lvl - f_lvl) ** 2, axis=1)
            rmsse      = np.sqrt(mse / info["scales"])
            level_scores.append(np.sum(rmsse * info["weights"]))


        report = pd.DataFrame({
            "Level": range(1, 13),
            "WRMSSE": level_scores,
            "Description": [str(self.levels[i]) for i in range(1, 13)]
        })
        return report
    
    def evaluate_all(self, forecast_df: pd.DataFrame, actual_df: pd.DataFrame) -> dict:
        """
        Compute WRMSSE across all 12 hierarchy levels plus convenience L12 metrics.

        Parameters
        ----------
        forecast_df : must contain columns [id, date, <forecast_value>]
                      forecast value column must be named self.target
        actual_df   : must contain columns [id, date, self.target]
        """


        # filter 
        actual_df = actual_df[(actual_df["date"]>=forecast_df["date"].min()) & (actual_df["date"]<=forecast_df["date"].max())].copy()
        #actual_df["id"] = actual_df["id"].astype(self.ids.dtype)
        forecast_df["id"] = forecast_df["id"].astype(self.ids.dtype)

        # ── Pivot ────────────────────────────────────────────────────────
        # Use self.target explicitly — never rely on column position
        f_wide = forecast_df.pivot(index="id", columns="date", values=self.target)
        a_wide = actual_df.pivot(  index="id", columns="date", values=self.target)

        f_wide.index = f_wide.index.astype(str) # to ensure alignment with self._static, which is also str-indexed
        a_wide.index = a_wide.index.astype(str) # to ensure alignment with self._static, which is also str-indexed

        common_dates = a_wide.columns.intersection(f_wide.columns)

        if len(common_dates) != 28:
            warnings.warn(f"Standard M5 evaluation is 28 days. You are evaluating {len(common_dates)} days.")

        # Warn loudly on partial overlap — silent data loss is worse than a warning
        if len(common_dates) < len(a_wide.columns):
            missing = len(a_wide.columns) - len(common_dates)
            print(f"[M5Evaluator] WARNING: {missing} actual date(s) have no matching forecast — treated as 0.")
        if len(common_dates) == 0:
            raise ValueError("forecast_df and actual_df share no common dates.")

        # Validate series alignment — reindex fills with 0 silently, so check first
        missing_ids = set(self.ids) - set(f_wide.index)
        if missing_ids:
            print(f"[M5Evaluator] WARNING: {len(missing_ids)} series missing from forecast — filled with 0.")

        # Align to training IDs and common dates; fillna(0) only after explicit warning above
        f12 = f_wide.reindex(index=self.ids, columns=common_dates).fillna(0)
        a12 = a_wide.reindex(index=self.ids, columns=common_dates).fillna(0)

        # Join static once; use explicit column names from here on — no positional slicing
        f12 = f12.join(self._static)
        a12 = a12.join(self._static)
        assert a12[self._static_cols].notna().any().any(), "Static join produced all NaN — index dtype mismatch between a_wide and self._static"
        date_cols = list(common_dates)

        # ── WRMSSE across 12 levels ──────────────────────────────────────
        level_scores = []
        for lvl in range(1, 13):
            info = self.level_info[lvl]

            # Aggregate using explicit date_cols — immune to column order changes
            f_lvl = self._aggregate(f12, date_cols, info["cols"], lvl).values
            a_lvl = self._aggregate(a12, date_cols, info["cols"], lvl).values

            mse        = np.mean((a_lvl - f_lvl) ** 2, axis=1)
            rmsse      = np.sqrt(mse / info["scales"])
            level_scores.append(np.sum(rmsse * info["weights"]))

        # ── L12 convenience metrics (on raw numpy arrays — no pandas overhead) ──
        f12_vals = f12[date_cols].values
        a12_vals = a12[date_cols].values


        return {
            "WRMSSE":   float(np.mean(level_scores)),
            "WAPE_L12": float(np.sum(np.abs(a12_vals - f12_vals)) / (np.sum(a12_vals) + 1e-8)),
            "RMSE_L12": float(np.sqrt(np.mean((a12_vals - f12_vals) ** 2))),
            "MAE_L12":  float(np.mean(np.abs(a12_vals - f12_vals))),
        }



In [ ]:
evaluator = M5Evaluator(train_df=hist_df, static_df=static_df,target_col=pipeline.target)

metrics1 = evaluator.evaluate_all(level_forecast_df_ag, df_actual_eval)

print(f"ETS WRMSSE: {metrics1['WRMSSE']:.4f} | WAPE: {metrics1['WAPE_L12']:.2%}")


--- Pre-calculating Hierarchy Scales and Weights ---
ETS WRMSSE: 0.9583 | WAPE: 78.11%


In [ ]:
from m5_benchmarks import M5BenchmarkSuite

suite = M5BenchmarkSuite(horizon=28, n_jobs=-1)

method = "ES_bu"
# Or run a single method
naive_fcst = suite.run(train_df=hist_df, methods=[method])[method]

evaluator = M5Evaluator(train_df=hist_df, static_df=static_df,target_col=pipeline.target)

metrics1 = evaluator.evaluate_all(naive_fcst, df_actual_eval)

print(f"ETS WRMSSE: {metrics1['WRMSSE']:.4f} | WAPE: {metrics1['WAPE_L12']:.2%}")


--- Running ES_bu via AutoGluon (statsforecast backend) ---


Renaming existing column 'item_id' -> '__item_id' to avoid name collisions.


OSError: [Errno 28] No space left on device